# KL Divergence Evaluation for Quantized LLMs

This notebook demonstrates the `KLDivergenceEvaluator` from `llmcompressor.evaluation`.

## Why KL Divergence?

After quantizing a model, we want to measure how much the output distribution has
shifted relative to the baseline. KL Divergence (KLD) quantifies this:

- **KLD ≈ 0** → quantized model outputs nearly identical distributions to baseline
- **KLD >> 0** → significant distribution shift; quantization quality may be poor

## How it works (efficient hidden-state approach)

Naively computing KLD via vLLM requires extracting full vocabulary logprobs
(`vocab_size ≈ 120k`), which is extremely slow (~64 hours for WikiText on Llama-3-8B).

Instead, we:
1. Hook into each model's `lm_head` via `LLM.apply_model` to capture **pre-projection
   hidden states** (`hidden_dim ≈ 4096`) — ~30x smaller than full logprobs.
2. Move hidden states to CPU.
3. Apply the baseline `lm_head` to both sets of hidden states offline.
4. Compute KLD from the resulting log-probabilities.

**Runtime**: minutes instead of hours.


## 1. Setup

> **Runtime**: Select `Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
# Install vllm and datasets from PyPI, then install llmcompressor from the
# branch containing the KLD evaluation module (not the published PyPI release).
# Replace YOUR_USERNAME and YOUR_BRANCH with your fork details before running.
!pip install vllm datasets -q
!pip install "git+https://github.com/YOUR_USERNAME/llm-compressor.git@YOUR_BRANCH" -q

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not detected — change runtime to GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Quantize a model with llmcompressor

We use `facebook/opt-125m` as a lightweight example. Replace with any HuggingFace
causal LM (e.g. `meta-llama/Meta-Llama-3-8B`) — just ensure you have access.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

BASE_MODEL_ID = "facebook/opt-125m"
QUANTIZED_SAVE_DIR = "opt-125m-W4A16"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

recipe = QuantizationModifier(
    targets="Linear",
    scheme="W4A16",
    ignore=["lm_head"],
)

oneshot(
    model=model,
    dataset="open_platypus",
    recipe=recipe,
    max_seq_length=512,
    num_calibration_samples=128,
)

model.save_pretrained(QUANTIZED_SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(QUANTIZED_SAVE_DIR)
print(f"Quantized model saved to: {QUANTIZED_SAVE_DIR}")

## 3. Evaluate KL Divergence

Compare the baseline and quantized model output distributions.

In [ ]:
from llmcompressor.evaluation import evaluate_kl_divergence

# Custom calibration prompts — use more for a reliable estimate
prompts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models require large amounts of training data.",
    "Quantization reduces model size by lowering the precision of weights.",
    "Natural language processing has advanced significantly in recent years.",
    "Efficient inference is critical for deploying large language models.",
    "The transformer architecture revolutionized sequence modeling tasks.",
    "Gradient descent optimizes model parameters by minimizing a loss function.",
    "Attention mechanisms allow models to focus on relevant input tokens.",
]

result = evaluate_kl_divergence(
    base_model_id=BASE_MODEL_ID,
    quantized_model_id=QUANTIZED_SAVE_DIR,
    prompts=prompts,
    dtype="auto",
    max_tokens=1,
    gpu_memory_utilization=0.85,
)

print(result)

## 4. Inspect results

In [ ]:
print(f"Mean KLD (base vs W4A16):  {result.mean_kld:.6f}")
print(f"Prompts evaluated:         {result.num_prompts}")
print(f"Total tokens evaluated:    {result.num_tokens}")
print()
print("Per-prompt KLD:")
for i, (prompt, kld) in enumerate(zip(prompts, result.per_prompt_kld)):
    print(f"  [{i}] {kld:.6f}  |  {prompt[:60]}")

## 5. Sanity check — same model should give KLD ≈ 0

In [ ]:
from llmcompressor.evaluation import evaluate_kl_divergence

sanity = evaluate_kl_divergence(
    base_model_id=BASE_MODEL_ID,
    quantized_model_id=BASE_MODEL_ID,  # same model
    prompts=prompts[:3],
)
print(f"Sanity check KLD (same model): {sanity.mean_kld:.8f}  (should be ~0.0)")

## 6. (Optional) WikiText evaluation

For a more thorough evaluation, use WikiText-2 calibration sentences.
Increase `num_samples` for a more reliable mean KLD estimate.

In [ ]:
from datasets import load_dataset

NUM_SAMPLES = 64  # increase for better coverage; ~512 recommended for production
MAX_CHARS = 512

wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_prompts = [
    row["text"][:MAX_CHARS].strip()
    for row in wikitext
    if len(row["text"].strip()) > 50
][:NUM_SAMPLES]

print(f"Evaluating on {len(wiki_prompts)} WikiText samples...")

wiki_result = evaluate_kl_divergence(
    base_model_id=BASE_MODEL_ID,
    quantized_model_id=QUANTIZED_SAVE_DIR,
    prompts=wiki_prompts,
)

print(f"WikiText Mean KLD: {wiki_result.mean_kld:.6f}")
print(f"Tokens evaluated:  {wiki_result.num_tokens}")

## 7. Using the class API directly

For repeated evaluations (e.g., comparing multiple quantization schemes),
use `KLDivergenceEvaluator` directly to avoid re-instantiating the evaluator.

In [ ]:
from llmcompressor.evaluation import KLDivergenceEvaluator

evaluator = KLDivergenceEvaluator(
    base_model_id=BASE_MODEL_ID,
    quantized_model_id=QUANTIZED_SAVE_DIR,
    dtype="auto",
    gpu_memory_utilization=0.85,
)

result = evaluator.evaluate(prompts=prompts)
print(f"Mean KLD: {result.mean_kld:.6f}")